# **FullGrad, NoiseGrad и FusionGrad: смещения сети и шум в весах**

Практика к модулю [«Эвристические объяснения на основе градиента»](https://ai-interpretability.school/learn/10/0/0).

Урок говорит о трех методах три вещи, и каждую можно проверить числом:

1. у сети из ReLU и батч-нормализации выход раскладывается **точно** на произведение градиента на вход и вклады всех смещений, и вклад смещений у такой сети «немаленький» (FullGrad, Srinivas и Fleuret, 2019);
2. карта FullGrad этого разложения не сохраняет: модуль каждого слагаемого берется до сложения, и сумма карты уже не равна выходу;
3. NoiseGrad усредняет карты по копиям модели с мультипликативным шумом в весах, FusionGrad добавляет к этому шум во входе, а уровень шума весов авторы выбирают так, чтобы точность модели просела примерно на 5 % (Bykov et al., 2022).

Модель и снимок те же, что в тетради `PRACTICE_smoothgrad_sanity`: ResNet-50 с весами ImageNet и фотография кабана из репозитория курса. Тетрадь считается на процессоре за несколько минут; FullGrad берем из библиотеки `grad-cam`, NoiseGrad и FusionGrad пишем сами, по формулам урока «SmoothGrad».

In [ ]:
!pip install -q grad-cam==1.5.7

In [ ]:
import copy
import time
import urllib.request
from io import BytesIO

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import spearmanr
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

torch.manual_seed(0)   # весь шум ниже идет от своих генераторов с фиксированным сидом
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/'

model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).eval()
for m in model.modules():          # ReLU без записи на месте: иначе он затирает выход батч-нормализации
    if isinstance(m, torch.nn.ReLU):
        m.inplace = False

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def load(name):
    return transform(Image.open(BytesIO(urllib.request.urlopen(DATA + name, timeout=30).read())).convert('RGB')).unsqueeze(0)


x = load('hog.jpg')
categories = urllib.request.urlopen(DATA + 'imagenet_classes.txt', timeout=30).read().decode().splitlines()
with torch.no_grad():
    logits = model(x)
target = int(logits.argmax())
print('класс:', target, categories[target], '  логит:', round(logits[0, target].item(), 4))

**Одна правка модели.** ReLU в `torchvision` пишет результат на место своего входа (`inplace=True`), а вход ReLU здесь — выход батч-нормализации. Для разложения в разделе 1 нужен градиент именно по выходу батч-нормализации, поэтому запись на месте выключена. Прогноз от этого не меняется: меняется только то, где хранится промежуточный тензор.

## 1. Тождество FullGrad: куда уходит логит

Урок записывает тождество для сети со смещениями $b$:

$$f(x; b) = \nabla_x f(x; b)^{\top} x + \nabla_b f(x; b)^{\top} b$$

Оно точное для сети, которая кусочно-линейна по входу и смещениям: ReLU, max-pooling, свертки, сложения в остаточных связях и батч-нормализация в режиме `eval()`. Все это про ResNet-50.

У сверток ResNet-50 явных смещений нет (`bias=False`), поэтому смещения сети — это **неявные смещения батч-нормализации** и смещение последнего линейного слоя. В режиме `eval()` батч-нормализация канала $c$ — это аффинное преобразование $\gamma_c \frac{z - \mu_c}{\sqrt{\sigma^2_c + \varepsilon}} + \beta_c$, и ее смещение

$$b_c = \beta_c - \frac{\mu_c \gamma_c}{\sqrt{\sigma^2_c + \varepsilon}},$$

где $\mu_c$ и $\sigma^2_c$ — бегущие среднее и дисперсия, накопленные при обучении. Это и есть «implicit biases, such as running averages of batch norm layers» из статьи. Вклад слоя батч-нормализации в логит — сумма по каналам и позициям произведения градиента логита по выходу слоя на $b_c$.

In [ ]:
bns = [m for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d)]


def bn_bias(m):
    # неявное смещение батч-нормализации в режиме eval()
    return (m.bias - m.running_mean * m.weight / torch.sqrt(m.running_var + m.eps)).detach()


def decompose(model, x, target):
    # Слагаемые тождества FullGrad: вход, каждый слой батч-нормализации, смещение fc.
    outputs = []

    def keep(module, inp, out):
        out.retain_grad()
        outputs.append(out)

    hooks = [m.register_forward_hook(keep) for m in bns]
    xg = x.clone().requires_grad_(True)
    logit = model(xg)[0, target]
    logit.backward()
    for h in hooks:
        h.remove()
    input_term = (xg.grad * x).sum().item()
    bn_terms = [(o.grad * bn_bias(m)[None, :, None, None]).sum().item() for m, o in zip(bns, outputs)]
    return logit.item(), input_term, bn_terms, model.fc.bias[target].item(), xg.grad.detach()


logit, input_term, bn_terms, fc_term, grad_x = decompose(model, x, target)
print(f'слоев батч-нормализации: {len(bns)}')
print(f'градиент на вход (Gradient×Input): {input_term:9.4f}')
print(f'смещения батч-нормализации:        {sum(bn_terms):9.4f}')
print(f'смещение fc:                       {fc_term:9.4f}')
print(f'сумма слагаемых:                   {input_term + sum(bn_terms) + fc_term:9.4f}')
print(f'логит класса:                      {logit:9.4f}')

Сумма слагаемых совпадает с логитом во всех четырех напечатанных знаках после точки: тождество держится, и ничего не выброшено.

**Задание 1.** Какую долю логита класса дает слагаемое Gradient×Input, то есть отношение `input_term / logit`? Какая доля приходится на все смещения вместе (батч-нормализация и `fc`)? Округлите обе доли до тысячных и сравните с утверждением урока, что у сети с батч-нормализацией выброшенная Gradient×Input часть выхода — «не поправка, а существенная доля».

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 2. Карта FullGrad: модуль до сложения

Карта из урока поднимает каждое слагаемое до размера входа и складывает их модули:

$$S_f(x) = \psi\big(\nabla_x f(x) \odot x\big) + \sum_c \psi\big(f^b(x)_c\big),$$

где $\psi$ — взятие модуля, нормировка и билинейная интерполяция до размера входа. Класс `FullGrad` в `grad-cam` устроен так же: он сам находит все слои батч-нормализации и сверточные слои со смещением, а слой выбирать не нужно, поэтому `target_layers` пустой. Смещение последнего линейного слоя в карту не входит: у него нет пространственной структуры.

Сначала посмотрим, сколько «массы» у слагаемых до нормировки: сумма модулей слагаемых против самого логита.

In [ ]:
from pytorch_grad_cam import FullGrad
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

print(f'сумма модулей слагаемых: {abs(input_term) + sum(abs(t) for t in bn_terms) + abs(fc_term):8.2f}')
print(f'модуль логита:           {abs(logit):8.2f}')

with FullGrad(model=model, target_layers=[]) as fullgrad:
    fullgrad_map = fullgrad(input_tensor=x, targets=[ClassifierOutputTarget(target)])[0]
    print(f'слоев со смещением у FullGrad: {len(fullgrad.target_layers)}')
print(f'карта FullGrad: размер {fullgrad_map.shape}, минимум {fullgrad_map.min():.2f}, '
      f'максимум {fullgrad_map.max():.2f}, сумма {fullgrad_map.sum():.1f}')

gxi_map = (grad_x * x).abs().sum(1)[0].numpy()        # модуль Gradient×Input, каналы сложены
vanilla_map = grad_x.abs().max(1).values[0].numpy()   # Vanilla Gradients из тетради SmoothGrad

fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
for a, (m, t) in zip(ax, [(vanilla_map, 'Vanilla Gradients'), (gxi_map, '|Gradient×Input|'), (fullgrad_map, 'FullGrad')]):
    a.imshow(m, cmap='hot')
    a.set_title(t)
    a.axis('off')
plt.tight_layout()
plt.show()

Сумма модулей слагаемых на порядок больше логита: слагаемые разных знаков при сложении гасят друг друга, а модуль это гашение убирает. Поэтому сумма карты FullGrad к логиту отношения не имеет, тем более после нормировки в $[0, 1]$. «Ничего не выброшено» относится к разложению, а не к картинке.

**Задание 2.** Насколько карта FullGrad похожа на карту одного своего слагаемого? Посчитайте корреляцию Спирмена между `fullgrad_map` и `gxi_map` и между `fullgrad_map` и `vanilla_map` (карты вытяните в вектор). Округлите до тысячных. Что говорят эти числа о том, откуда берется рисунок карты FullGrad?

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 3. NoiseGrad: шум в весах и выбор его уровня

NoiseGrad умножает **все** обученные веса сети на гауссов шум со средним единица и усредняет карты базового метода по $M$ таким копиям:

$$W_i = \hat{W} \cdot \eta_i, \quad \eta_i \sim N(\mathbf{1},\ \sigma^2_{NG} I), \qquad E_{NG}(x) = \frac{1}{M}\sum_{i=1}^{M} E\big(x,\ f(\cdot,\ W_i)\big)$$

Базовый метод $E$ здесь — Vanilla Gradients с той же сверткой каналов, что в тетради `PRACTICE_smoothgrad_sanity`: сначала модуль, потом максимум по каналам. Шум накладывается на все параметры модели (`model.parameters()`), включая $\gamma$ и $\beta$ батч-нормализации; бегущие средние — не параметры, а буферы, и шум их не трогает.

Уровень $\sigma_{NG}$ авторы предлагают поднимать, пока точность модели не просядет примерно на 5 %. Размеченной выборки ImageNet у нас нет, поэтому возьмем замену подешевле: шесть снимков из папки `data` репозитория курса и долю случаев, где зашумленная копия выдает тот же класс, что исходная модель. Шесть снимков на десяти копиях — 60 прогнозов, это грубая мера, и ее грубость стоит держать в голове.

In [ ]:
def vanilla_gradient(model, x, target):
    x = x.clone().requires_grad_(True)
    grad, = torch.autograd.grad(model(x)[0, target], x)
    return grad


def to_map(grad):
    return grad.abs().max(dim=1).values[0]


def noisy_model(model, sigma, gen):
    # Копия сети, все параметры которой умножены на шум N(1, sigma^2).
    noisy = copy.deepcopy(model)
    with torch.no_grad():
        for p in noisy.parameters():
            p.mul_(1 + sigma * torch.randn(p.shape, generator=gen))
    return noisy.eval()


def noisegrad(model, x, target, sigma, M=10, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return torch.stack([to_map(vanilla_gradient(noisy_model(model, sigma, gen), x, target)) for _ in range(M)]).mean(0)


def spearman(a, b):
    return spearmanr(np.ravel(a), np.ravel(b)).statistic


probe = torch.cat([load(n) for n in ('hog.jpg', 'cat.jpg', 'cat_and_dog.jpg', 'pig.png',
                                     'geranimo-ADUi-P4n-Jwds-unsplash.jpg', 'glen-carrie-4-Qr-Cip9gbx-Q-unsplash.jpg')])
with torch.no_grad():
    probe_classes = model(probe).argmax(1)
print('классы исходной модели на шести снимках:', [categories[c] for c in probe_classes])

vanilla = to_map(vanilla_gradient(model, x, target))
SIGMAS = (0.005, 0.01, 0.02, 0.05)
print(f'\n{"σ_NG":>6} {"совпало прогнозов":>18} {"Спирмен с Vanilla":>18} {"Спирмен двух сидов":>19}')
for sigma in SIGMAS:
    gen = torch.Generator().manual_seed(0)
    agree = 0
    for _ in range(10):
        with torch.no_grad():
            agree += int((noisy_model(model, sigma, gen)(probe).argmax(1) == probe_classes).sum())
    a, b = noisegrad(model, x, target, sigma, seed=1), noisegrad(model, x, target, sigma, seed=2)
    print(f'{sigma:6.3f} {agree:>11} из 60 {spearman(vanilla, a):18.3f} {spearman(a, b):19.3f}')

Столбцы таблицы: сколько из 60 прогнозов зашумленных копий совпало с прогнозом исходной модели; корреляция Спирмена карты NoiseGrad ($M = 10$) с картой Vanilla Gradients; корреляция Спирмена двух карт NoiseGrad, посчитанных с разными сидами шума, то есть устойчивость самого NoiseGrad.

**Задание 3.** Правило авторов здесь звучит так: совпасть должны не меньше 95 % прогнозов, то есть 57 из 60. Какой самый большой $\sigma_{NG}$ из таблицы проходит это правило? Чему при нем равна корреляция Спирмена NoiseGrad с Vanilla (до тысячных)? Что это значит для того, насколько NoiseGrad при таком шуме меняет карту?

**Ваш ответ:**

## 4. FusionGrad и сравнение с SmoothGrad

FusionGrad — это SmoothGrad, вложенный в NoiseGrad: на каждой из $M$ зашумленных копий модели карта усредняется по $N$ зашумленным копиям входа.

$$E_{FG}(x) = \frac{1}{N}\sum_{i=1}^{N} \frac{1}{M}\sum_{j=1}^{M} E\big(x + \xi_i,\ f(\cdot,\ W_j)\big)$$

Уровень шума во входе берем тот же, что в тетради `PRACTICE_smoothgrad_sanity`: 0.15 от размаха значений входа. Уровень шума весов — $\sigma_{NG} = 0.005$, тот, что прошел правило авторов в разделе 3. Бюджет прогонов у всех методов, кроме FusionGrad, — 10 карт; у FusionGrad $N = M = 5$, то есть 25 карт.

Ячейка ниже считает четыре карты, корреляции Спирмена между ними, устойчивость каждой (корреляцию двух карт с разными сидами), число обратных проходов и время на этой машине.

In [ ]:
SIGMA_NG, SIGMA_SG = 0.005, 0.15


def smoothgrad(model, x, target, n=10, seed=0):
    gen = torch.Generator().manual_seed(seed)
    s = SIGMA_SG * (x.max() - x.min())
    return torch.stack([to_map(vanilla_gradient(model, x + s * torch.randn(x.shape, generator=gen), target))
                        for _ in range(n)]).mean(0)


def fusiongrad(model, x, target, N=5, M=5, seed=0):
    gen = torch.Generator().manual_seed(seed)
    s = SIGMA_SG * (x.max() - x.min())
    maps = []
    for _ in range(M):
        noisy = noisy_model(model, SIGMA_NG, gen)
        for _ in range(N):
            maps.append(to_map(vanilla_gradient(noisy, x + s * torch.randn(x.shape, generator=gen), target)))
    return torch.stack(maps).mean(0)


methods = {
    'Vanilla':    (lambda seed: vanilla, 1),
    'SmoothGrad': (lambda seed: smoothgrad(model, x, target, seed=seed), 10),
    'NoiseGrad':  (lambda seed: noisegrad(model, x, target, SIGMA_NG, seed=seed), 10),
    'FusionGrad': (lambda seed: fusiongrad(model, x, target, seed=seed), 25),
}
maps, stability = {}, {}
print(f'{"метод":12}{"проходов":>10}{"секунд":>9}{"устойчивость":>14}')
for name, (run, passes) in methods.items():
    start = time.time()
    maps[name] = run(1)
    seconds = time.time() - start
    stability[name] = spearman(maps[name], run(2))
    print(f'{name:12}{passes:>10}{seconds:9.1f}{stability[name]:14.3f}')

names = list(maps)
print('\nкорреляция Спирмена между картами')
print(' ' * 12 + ''.join(f'{n:>12}' for n in names))
for a in names:
    print(f'{a:12}' + ''.join(f'{spearman(maps[a], maps[b]):12.3f}' for b in names))

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, n in zip(ax, names):
    a.imshow(maps[n].numpy(), cmap='hot')
    a.set_title(n)
    a.axis('off')
plt.tight_layout()
plt.show()

Устойчивость Vanilla равна единице по построению: в ней нет шума. Время зависит от машины; число проходов — нет, и сравнивать стоимость методов лучше по нему.

**Задание 4.** К какой карте FusionGrad ближе по корреляции Спирмена — к SmoothGrad или к NoiseGrad? Объясните, почему так вышло, опираясь на результат задания 3. Изменится ли ответ, если поднять $\sigma_{NG}$ до 0.05? Проверьте: пересчитайте FusionGrad с `SIGMA_NG = 0.05` и сравните корреляции. Не забудьте вернуть `SIGMA_NG = 0.005` для следующего раздела.

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 5. Где ломается: проверка рандомизацией весов

NoiseGrad сам шевелит веса, поэтому естественный вопрос — проходит ли он проверку рандомизацией весов (Adebayo et al., 2018) из раздела 4 тетради `PRACTICE_smoothgrad_sanity`. Повторим каскад: переинициализируем слои сети сверху вниз и смотрим корреляцию Спирмена новой карты с картой исходной сети. Карта, которая объясняет модель, должна заметно меняться, когда модель сломана.

In [ ]:
def randomize(model, blocks, seed=0):
    # Копия сети со случайно переинициализированными весами перечисленных блоков.
    torch.manual_seed(seed)
    broken = copy.deepcopy(model)
    for name in blocks:
        for m in getattr(broken, name).modules():
            if isinstance(m, (torch.nn.Conv2d, torch.nn.Linear, torch.nn.BatchNorm2d)):
                m.reset_parameters()
    return broken.eval()


cascade = [['fc'], ['fc', 'layer4'], ['fc', 'layer4', 'layer3'], ['fc', 'layer4', 'layer3', 'layer2']]
print(f'{"переинициализировано":32}{"Vanilla":>10}{"NoiseGrad":>11}')
for blocks in cascade:
    broken = randomize(model, blocks)
    v = spearman(vanilla, to_map(vanilla_gradient(broken, x, target)))
    n = spearman(maps['NoiseGrad'], noisegrad(broken, x, target, SIGMA_NG, seed=1))
    print(f'{", ".join(blocks):32}{v:10.3f}{n:11.3f}')

**Задание 5.** Проходит ли NoiseGrad проверку так же, как Vanilla Gradients? Сравните два столбца и объясните сходство или различие, опираясь на то, насколько NoiseGrad при $\sigma_{NG} = 0.005$ отличается от Vanilla.

**Ваш ответ:**

## Что унести из тетради

- **Тождество FullGrad точное.** У ResNet-50 сумма произведения градиента на вход и вкладов неявных смещений батч-нормализации совпадает с логитом во всех четырех напечатанных знаках после точки.
- **Вклад смещений бывает больше всего логита.** На этом снимке произведение градиента на вход отрицательно, и весь логит, с запасом, дают смещения. Метод, который их выбрасывает, раскладывает не тот выход.
- **Карта FullGrad — не разложение.** Модуль берется до сложения, сумма модулей на порядок больше логита, а рисунок карты задают вклады смещений, а не градиент на вход.
- **Уровень шума весов NoiseGrad выбирают по модели, а не по вкусу.** На ResNet-50 правило «точность падает не больше чем на 5 %» оставляет такой малый шум, что карта NoiseGrad почти совпадает с Vanilla Gradients, а FusionGrad — со SmoothGrad.
- **Стоимость считают в проходах.** SmoothGrad и NoiseGrad — по $n$ или $M$ проходов, FusionGrad — $N \times M$.